# 12 · The suit's senses – OCR and Read

Not everything is a language model. Tony's suit sees, reads and measures. In Foundry, the **same Foundry resource** that hosts your models also offers the **Foundry Tools** (formerly Azure AI Services): purpose-built services for documents, images, speech and more. This chapter reads text out of pictures and PDFs.

| Tool (SDK package) | Best for | Input |
|---|---|---|
| **Azure AI Vision – Read** (`azure-ai-vision-imageanalysis`) | fast OCR of photos, signs, whiteboards | images |
| **Document Intelligence – `prebuilt-read`** (`azure-ai-documentintelligence`) | OCR of documents: PDFs, scans, multi-page, handwriting, languages | PDF, images, Office |
| **Document Intelligence – `prebuilt-layout`** | text **plus structure**: tables, headings, selection marks, as Markdown | PDF, images, Office |
| **Content Understanding** (`azure-ai-contentunderstanding`) | LLM-powered extraction with your own schema (chapter 13) | documents, images, audio, video |

All of them use **one endpoint** (`STARK_ACCOUNT_ENDPOINT`), **Entra ID** (no keys) and the *Cognitive Services User* role that `azd up` gave you. Deep dive: [docs/foundry-tools.md](../docs/foundry-tools.md). Sample files come from [scripts/make_sample_docs.py](../scripts/make_sample_docs.py); all content is fictional.

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_docs import *

print("Foundry resource endpoint:", cfg("STARK_ACCOUNT_ENDPOINT"))
from IPython.display import Image as Show, Markdown
Show(filename=str(ROOT / "data" / "docs" / "whiteboard-workshop.png"))

## 1 · Vision Read on a photo
The whiteboard is slightly rotated, like a real photo. Read returns **lines** with a bounding polygon and per-word confidence. `vision_read()` ([stark_docs.py](stark_docs.py)) makes the call and returns each line as `text`, `polygon` and the average word `confidence`.

**Expected:** the title `WORKSHOP TODO` and the five lines below it (written by [make_sample_docs.py](../scripts/make_sample_docs.py)), each with a confidence between 0 and 1. ❓ Read may split or join a line differently.

In [ ]:
whiteboard_lines = vision_read("data/docs/whiteboard-workshop.png")
for line in whiteboard_lines:
    print(f"{line['confidence']:.2f}  {line['text']}")

### See what it found
Drawing the polygons is the fastest way to debug OCR. The cell draws `whiteboard_lines` from the cell above onto the photo.

In [ ]:
from PIL import Image, ImageDraw
whiteboard = Image.open(ROOT / "data" / "docs" / "whiteboard-workshop.png").convert("RGB")
draw = ImageDraw.Draw(whiteboard)
for line in whiteboard_lines:
    draw.polygon(line["polygon"], outline="red", width=3)
whiteboard

## 2 · Document Intelligence `prebuilt-read` on a PDF
Documents are different from photos: several pages, fonts, languages, maybe handwriting. `AnalyzeDocumentRequest(bytes_source=...)` sends the file; the call is a long-running operation (`begin_...` + `.result()`). `di_analyze()` ([stark_docs.py](stark_docs.py)) wraps both.

**Expected:** `pages: 1` and `handwritten spans: 0`: the invoice is one typed page. The first lines start with the vendor, `Pym Particle Supply Co.`. ❓ The detected language should be `en`; the list can also be empty.

In [ ]:
read_result = di_analyze("prebuilt-read", "data/docs/invoice-pym-4471.pdf")
print("pages:", len(read_result.pages), "| languages:", [(language.locale, round(language.confidence, 2)) for language in (read_result.languages or [])])
print("handwritten spans:", sum(1 for style in (read_result.styles or []) if style.is_handwritten))
print("---- first lines ----")
for line in read_result.pages[0].lines[:12]:
    print(line.content)

## 3 · `prebuilt-layout`: text *and* structure
For an invoice the **table** matters. Layout returns tables as cells with row/column indexes, or the whole document as **Markdown** – ideal input for an LLM. The cell shows the whole Markdown, uncut, as an LLM would get it.

**Expected:** `tables found: 1`, the line-item table. ❓ In the Markdown, Layout writes that table as an HTML `<table>`.

In [ ]:
layout_result = di_analyze("prebuilt-layout", "data/docs/invoice-pym-4471.pdf", output_content_format="markdown")
print("tables found:", len(layout_result.tables or []))
display(Markdown(f"```markdown\n{layout_result.content}\n```"))

The first table as a DataFrame, built from its cells.

**Expected:** the columns `Description`, `Qty`, `Unit price`, `Amount`; one row each for the Pym particles (4 × $450.00 = $1,800.00), the shrink/grow regulator ($1,200.00) and shipping ($85.50); then `Subtotal` $3,085.50, `Tax (8%)` $246.84 and `Total due` $3,332.34.

In [ ]:
import pandas as pd
invoice_table = layout_result.tables[0]
grid = [[""] * invoice_table.column_count for _ in range(invoice_table.row_count)]
for cell in invoice_table.cells:
    grid[cell.row_index][cell.column_index] = cell.content
pd.DataFrame(grid[1:], columns=grid[0])

## 4 · Check the OCR like an engineer
Garbage in, garbage out. For documents you care about, **assert** a few known facts (a mini evaluation – chapter 10 for the big version).

**Goal:** check the Vision Read result of section 1 against five facts we know are on the whiteboard. The cell uses `whiteboard_lines`, so run section 1 first.

| Fact | Line on the whiteboard |
|---|---|
| `WORKSHOP` | WORKSHOP TODO |
| `arc reactor` | 1. Service arc reactor (Pepper present!) |
| `Mark 86` | 2. Mark 86 gauntlet: fit test Friday |
| `PO-7781` | 4. Order Pym particles (PO-7781) |
| `2026-10-30` | Owner: T. Stark   Due: 2026-10-30 |

**Expected:** all five ✅, ignoring case, and the `assert` passes. A ❌ means Read missed or misread that text: find its line in section 1 and its polygon in the picture.

In [ ]:
whiteboard_text = " ".join(line["text"] for line in whiteboard_lines)
expected_facts = ["WORKSHOP", "arc reactor", "Mark 86", "PO-7781", "2026-10-30"]
missing_facts = [fact for fact in expected_facts if fact.lower() not in whiteboard_text.lower()]
display(Markdown("\n".join(["| Fact | Found |", "|---|---|"] +
                            [f"| `{fact}` | {'❌' if fact in missing_facts else '✅'} |" for fact in expected_facts])))
assert not missing_facts, f"OCR missed: {missing_facts}"

## 🧹 Clean up
Nothing to delete: these calls analyse a file and store nothing in Azure.

## 🎯 Recap
| Situation | Pick |
|---|---|
| A photo or screenshot, just the text | Vision Read |
| PDF/scan, multi-page, languages, handwriting | Document Intelligence `prebuilt-read` |
| You need tables or reading order | `prebuilt-layout` (Markdown) |
| You need *fields* (vendor, total, ...) | prebuilt models (chapter 13) |
| Your own schema, messy or varied documents | Content Understanding (chapter 13) |

⚠️ Documents may contain personal data; Foundry Tools process it in the resource's region. Mind limits (file size, pages per request) and pricing per page.

> 💡 Next: [13 · Invoices and Content Understanding](13_senses_document_intelligence.ipynb).